In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Inno/data_2

In [ ]:
# !unzip -q sample_submission.zip

# !apt-get install zstd -y -q
# !zstd -3 -T0 yellow_tripdata_2015-01.csv -o yellow_tripdata_2015-01.zst
# !zstd -d -T0 yellow_tripdata_2015-01.zst -o yellow_tripdata_2015-01_mm.csv

# !tar --use-compress-program="zstd -3 -T0" -cf my_backup.tar.zst *
# !tar --use-compress-program="zstd -T0" -xf my_backup.tar.zst

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import plotly.express as px
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold, train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import TargetEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

#**EDA**

In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
df.info(memory_usage='deep', show_counts=True, verbose=True)

In [ ]:
categor_cols = df.select_dtypes(include=['object']).columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
df[numeric_cols].describe().loc[['count', 'min', 'max']]

In [ ]:
df[categor_cols].describe()

1. EDA

1.1) id - удалить колонку

1.2) vendor_id - превратить тип int64 в что-нибудь поменьше

1.3) dropoff_datetime - удалить, т.к. напрямую связанна через pickup_datetime с trip_duration (target)

1.4) passenger_count - уменьшить тип с int64 в что-нибудь поменьше

1.5) store_and_fwd_flag - превратить в бинарную переменную

2. Data Preparation

2.1) pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude -На их основе сгенерировть новые признаки: расчет расстояния, определение локаций, Метрика Манхэттена

2.2)

In [ ]:
# 1. EDA
if 'id' in df.columns:
    del df['id'] # 1.1
    df['vendor_id'] = df['vendor_id'].astype('uint8') # 1.2
    del df['dropoff_datetime'] # 1.3
    df['passenger_count'] = df['passenger_count'].astype('uint8') # 1.4
    df["store_and_fwd_flag"] = df["store_and_fwd_flag"].map({"N": 0, "Y": 1}).astype('uint8') # 1.5

#**Data Preparation**

In [ ]:
# 2. Data Preparation

In [ ]:
# 2.1

In [ ]:
# Дистанционные признаки
def calculate_distances(df):
    # Переводим координаты в радианы для формулы Гаверсинуса
    lat1, lon1 = np.radians(df['pickup_latitude']), np.radians(df['pickup_longitude'])
    lat2, lon2 = np.radians(df['dropoff_latitude']), np.radians(df['dropoff_longitude'])

    # 1. Расстояние Гаверсинуса (в километрах)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    df['distance_haversine_km'] = 6371 * c

    # 2. Манхэттенское расстояние (в километрах)
    # Аппроксимация: 1 градус широты ~ 111 км, 1 градус долготы ~ 111 км * cos(latitude)
    lat_dist = np.abs(df['pickup_latitude'] - df['dropoff_latitude']) * 111.0
    lon_dist = np.abs(df['pickup_longitude'] - df['dropoff_longitude']) * 111.0 * np.cos(np.radians(df['pickup_latitude']))
    df['distance_manhattan_km'] = lat_dist + lon_dist

    # 3. Евклидово расстояние (в градусах, полезно для деревьев)
    df['distance_euclidean_deg'] = np.sqrt((df['pickup_latitude'] - df['dropoff_latitude'])**2 +
                                           (df['pickup_longitude'] - df['dropoff_longitude'])**2)
    return df

df = calculate_distances(df)

In [ ]:
# Векторные и геометрические признаки
def calculate_geometry(df):
    # 1. Центроид поездки (средняя точка)
    df['centroid_latitude'] = (df['pickup_latitude'] + df['dropoff_latitude']) / 2
    df['centroid_longitude'] = (df['pickup_longitude'] + df['dropoff_longitude']) / 2

    # 2. Простые разности координат (направление смещения)
    df['delta_latitude'] = df['dropoff_latitude'] - df['pickup_latitude']
    df['delta_longitude'] = df['dropoff_longitude'] - df['pickup_longitude']

    # 3. Направление движения (Bearing) в градусах от 0 до 360
    lat1, lon1 = np.radians(df['pickup_latitude']), np.radians(df['pickup_longitude'])
    lat2, lon2 = np.radians(df['dropoff_latitude']), np.radians(df['dropoff_longitude'])

    d_lon = lon2 - lon1
    y = np.sin(d_lon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(d_lon)

    bearing = np.degrees(np.arctan2(y, x))
    df['direction_bearing'] = (bearing + 360) % 360

    return df

df = calculate_geometry(df)

In [ ]:
def find_optimal_clusters_full_data(all_coords, max_k=80):
    inertias = []
    # Шаг 10 оптимален: проверим 10, 20, 30 ... 150 кластеров
    k_values = range(10, max_k + 1, 10)

    for k in k_values:
        # Увеличили batch_size до 50 000, так как памяти достаточно
        kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=50000)
        kmeans.fit(all_coords)
        inertias.append(kmeans.inertia_)

    # Автоматический поиск точки перегиба («локтя»)
    diffs = np.diff(inertias)
    diffs_ratios = diffs[:-1] / diffs[1:]
    optimal_index = np.argmax(diffs_ratios) + 1
    optimal_k = k_values[optimal_index]

    return optimal_k

def apply_full_auto_clustering(df):
    all_coords = np.vstack([
        df[['pickup_latitude', 'pickup_longitude']].values,
        df[['dropoff_latitude', 'dropoff_longitude']].values
    ])

    # Ищем среди диапазона до 100
    n_clusters = find_optimal_clusters_full_data(all_coords, max_k=150)

    print(f"{n_clusters} кластеров")
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=30000)
    kmeans.fit(all_coords)

    df['pickup_cluster'] = kmeans.predict(df[['pickup_latitude', 'pickup_longitude']].values)
    df['dropoff_cluster'] = kmeans.predict(df[['dropoff_latitude', 'dropoff_longitude']].values)
    df['route_cluster_combo'] = df['pickup_cluster'].astype(str) + '_' + df['dropoff_cluster'].astype(str)

    return df

# Запуск полного цикла
# df = apply_full_auto_clustering(df)

In [ ]:
def visualize_clusters(df, n_samples=50000):
    # 1. Берем случайную подвыборку, чтобы карта не зависала
    sample_df = df.sample(n=min(n_samples, len(df)), random_state=42).copy()

    # 2. Переводим кластер в строковый тип, чтобы Plotly воспринимал его как категорию (разные цвета)
    sample_df['pickup_cluster'] = sample_df['pickup_cluster'].astype(str)

    # 3. Строим интерактивную карту точек начала поездки (Pickup)
    fig = px.scatter_mapbox(
        sample_df,
        lat="pickup_latitude",
        lon="pickup_longitude",
        color="pickup_cluster", # Точки окрасятся в цвет своего района
        color_discrete_sequence=px.colors.qualitative.Alphabet, # Набор ярких контрастных цветов
        zoom=10,
        height=700,
        title=f"Автоматические геозоны города (выборка {n_samples} точек)",
        labels={'pickup_cluster': 'ID Геозоны'}
    )

    # 4. Настраиваем бесплатную подложку карты (OpenStreetMap)
    fig.update_layout(
        mapbox_style="open-street-map",
        margin={"r":0,"t":40,"l":0,"b":0} # Убираем лишние отступы по краям
    )

    # 5. Отображаем карту (в Jupyter Notebook она откроется сама, в скрипте — в браузере)
    fig.show()

# visualize_clusters(df)

In [ ]:
if 'pickup_latitude' in df.columns:
    for s in ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude']:
        del df[s]

In [ ]:
# Подготовка к обучению (Удаление исходных колонок)
# Перед подачей в модели мы удаляем исходные координаты.
# *) Для Random Forest и XGBoost: колонку route_cluster_combo нужно перевести в category или использовать LabelEncoder.
# *) Для Линейной модели: категориальные признаки (pickup_cluster, dropoff_cluster, route_cluster_combo) необходимо закодировать через OneHotEncoder (get_dummies).

In [ ]:
# 2.2

In [ ]:
def generate_advanced_features(df):
    # 1. Разбор даты и времени
    df['hour'] = df['pickup_datetime'].dt.hour
    df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['minute_of_day'] = df['hour'] * 60 + df['pickup_datetime'].dt.minute

    # 2. Категории времени суток (полезно для линейной модели)
    # 0 - ночь, 1 - утро, 2 - день, 3 - вечер
    df['time_of_day'] = pd.cut(df['hour'], bins=[-1, 5, 11, 16, 23], labels=[0, 1, 2, 3]).astype(int)

    # # 4. СИНЕРГИЯ 1: Кластер маршрута + Час (Строковый признак для CatBoost/XGBoost)
    # df['route_cluster_combo'] = df['pickup_cluster'].astype(str) + '_' + df['dropoff_cluster'].astype(str)
    # df['route_hour_combo'] = df['route_cluster_combo'] + '_h' + df['hour'].astype(str)

    # 5. СИНЕРГИЯ 2: Ожидаемая скорость (Прокси-признак)
    # В часы пик скорость ниже. Создаем коэффициент «загруженности» часа.
    # Для линейной модели это даст нелинейную подсказку.
    # (Вы вычисляете среднюю скорость по часам на ТРЕЙНЕ, тут пример маппинга)
    rush_hour_map = {8: 15, 9: 15, 17: 12, 18: 12, 23: 40, 12: 25} # примерная скорость в км/ч
    df['expected_hourly_speed'] = df['hour'].map(rush_hour_map).fillna(25)

    # Расчет приблизительного времени на основе исторической скорости часа пик
    df['estimated_duration_by_speed'] = (df['distance_haversine_km'] / df['expected_hourly_speed']) * 60 # в минутах

    # ДОБАВИТЬ ЭТУ СТРОКУ: логарифмируем расчетное время для синергии с логом таргета
    df['estimated_duration_by_speed_log'] = np.log1p(df['estimated_duration_by_speed'])

    # 6. СИНЕРГИЯ 3: Вместимость вендора
    df['vendor_passenger_interaction'] = df['vendor_id'].astype(str) + '_p' + df['passenger_count'].astype(str)

    return df

# Переводим колонку в правильный формат даты и времени
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Теперь функция отработает без ошибок
df = generate_advanced_features(df)

In [ ]:
# Подготовка фичей под ваши модели:
# 1) Для Линейной модели (Baseline):
#     1.1) Удалите исходный pickup_datetime.
#     1.2) Примените One-Hot Encoding к колонкам: hour, day_of_week, time_of_day, vendor_id.
#     1.3) Важно: Признак estimated_duration_by_speed (оценочное время) станет сильнейшим линейным признаком для бейзлайна.
# 2) Для Random Forest и XGBoost:
#     2.1) Оставьте числовые: hour, day_of_week, minute_of_day, passenger_count, estimated_duration_by_speed.
#     2.2) Переведите текстовые синергетические колонки (route_hour_combo, vendor_passenger_interaction) в тип category (для XGBoost) или закодируйте через LabelEncoder.

#**TARGET ENCODING**

In [ ]:
# Выделяем признаки (X) и таргет (y)
X = df.drop(columns=['trip_duration'])
y = df['trip_duration']

# Разделяем на обучающую и тестовую выборки (например, 80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
class FastTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, n_splits=5, smoothing=10):
        self.n_splits = n_splits
        self.smoothing = smoothing
        self.global_mean = None
        self.mapping = None

    def fit(self, X, y):
        # Сохраняем глобальное среднее для новых категорий на тесте
        self.global_mean = y.mean()

        # Создаем временный датафрейм для быстрых группировок
        df_tmp = pd.DataFrame({'feat': X['route_hour_combo'].values, 'target': y.values})

        # Считаем финальные сглаженные средние для ТЕСТОВОЙ выборки
        stats = df_tmp.groupby('feat')['target'].agg(['count', 'mean'])
        # Формула сглаживания: (count * mean + smoothing * global_mean) / (count + smoothing)
        smoothed_vals = (stats['count'] * stats['mean'] + self.smoothing * self.global_mean) / (stats['count'] + self.smoothing)
        self.mapping = smoothed_vals.to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        # Для теста просто маппим уже готовый словарь (работает за доли секунды)
        # Если категория новая — подставляется global_mean
        X['TE_route_hour_duration'] = X['route_hour_combo'].map(self.mapping).fillna(self.global_mean)
        X = X.drop(columns=['route_hour_combo'], errors='ignore')
        return X

    def fit_transform(self, X, y):
        """
        Переопределяем fit_transform, чтобы применить ОЧЕНЬ быстрый Out-of-Fold на трейне.
        Вместо вызова полного цикла scikit-learn, делаем векторное разбиение.
        """
        self.fit(X, y)

        X_out = X.copy()
        df_tmp = pd.DataFrame({'feat': X['route_hour_combo'].values, 'target': y.values})
        oof_predictions = np.zeros(len(X))

        # Ручной K-Fold через векторизованный групбай — отработает за секунды
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)
        for train_idx, val_idx in kf.split(df_tmp):
            # Берем тренировочную часть фолда
            train_fold = df_tmp.iloc[train_idx]

            # Считаем сглаженное среднее внутри фолда
            stats = train_fold.groupby('feat')['target'].agg(['count', 'mean'])
            smoothed = (stats['count'] * stats['mean'] + self.smoothing * self.global_mean) / (stats['count'] + self.smoothing)
            fold_mapping = smoothed.to_dict()

            # Применяем к валидационной части фолда
            val_feats = df_tmp.iloc[val_idx]['feat']
            oof_predictions[val_idx] = val_feats.map(fold_mapping).fillna(self.global_mean).values

        X_out['TE_route_hour_duration'] = oof_predictions
        X_out = X_out.drop(columns=['route_hour_combo'], errors='ignore')
        return X_out

In [ ]:
# Шаг 1. Логарифмируем таргет
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Шаг 2. Запускаем Target Encoding на логарифмах (код из прошлого шага)
fast_te = FastTargetEncoder(n_splits=5, smoothing=10)
X_train['TE_route_hour_duration'] = fast_te.fit_transform(X_train[['route_hour_combo']], y_train_log)['TE_route_hour_duration']
X_test['TE_route_hour_duration'] = fast_te.transform(X_test[['route_hour_combo']])['TE_route_hour_duration']

# Шаг 3. Удаляем хвосты и текстовые колонки
cols_to_drop = ['pickup_datetime', 'route_cluster_combo', 'route_hour_combo', 'vendor_passenger_interaction']
X_train_clean = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test_clean = X_test.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
# +) Для Линейной модели (Baseline): Удалите текстовые колонки route_cluster_combo и route_hour_combo.
# Новая числовая колонка TE_route_hour_duration станет для неё главным ориентиром. Также не забудьте сделать
# One-Hot Encoding для простых категорий (hour, day_of_week, vendor_id).
# +) Для XGBoost и Random Forest: Вы можете либо удалить старый текстовый route_hour_combo
# (так как числовой аналог уже есть), либо перевести его в тип category (XGBoost умеет работать с ними напрямую в паре с числовым признаком).

#**ОБУЧЕНИЕ**

In [ ]:
# ОБУЧЕНИЕ

In [ ]:
# Категориальные признаки для кодирования (для Линейной модели)
cat_cols = ['vendor_id', 'hour', 'day_of_week', 'time_of_day']

In [ ]:
# Шаг 2. Подготовка данных для Линейной модели (Ridge / LinearRegression)

# Находим числовые колонки (все, кроме категориальных)
num_cols = [col for col in X_train_clean.columns if col not in cat_cols]

# Создаем препроцессор для линейной модели
lr_preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols), # Масштабируем числа (важно для регуляризации)
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols) # Кодируем категории в 0 и 1
])

# Собираем конвейер (Pipeline) для Baseline
lr_pipeline = Pipeline(steps=[
    ('preprocessor', lr_preprocessor),
    ('model', Ridge(alpha=1.0)) # Используем Ridge (линейная регрессия с защитой от переобучения)
])

# Обучаем линейную модель
lr_pipeline.fit(X_train_clean, y_train_log)
lr_preds_log = lr_pipeline.predict(X_test_clean)
print(f"Linear Baseline RMSLE: {root_mean_squared_error(y_test_log, lr_preds_log):.4f}")

In [ ]:
# Шаг 3. Подготовка и обучение Random Forest

# Для Random Forest мы берем чистые данные без One-Hot (деревья отлично работают с интами)
rf_model = RandomForestRegressor(n_estimators=24, max_depth=12, random_state=42, n_jobs=-1)

# Обучаем
rf_model.fit(X_train_clean, y_train_log)
rf_preds_log = rf_model.predict(X_test_clean)
print(f"Random Forest RMSLE: {root_mean_squared_error(y_test_log, rf_preds_log):.4f}")

In [ ]:
# Шаг 4. Подготовка и обучение XGBoost

# Создаем копии данных специально для XGBoost
X_train_xgb = X_train_clean.copy()
X_test_xgb = X_test_clean.copy()

# Явно указываем XGBoost, какие колонки считать категориями
for col in cat_cols:
    X_train_xgb[col] = X_train_xgb[col].astype('category')
    X_test_xgb[col] = X_test_xgb[col].astype('category')

# Инициализируем модель с поддержкой категорий (enable_categorical=True)
xgb_model = xgb.XGBRegressor(
    n_estimators=24,
    max_depth=24,
    learning_rate=0.05,
    enable_categorical=True, # Включаем нативную работу с категориями
    random_state=42,
    n_jobs=-1
)

# Обучаем
xgb_model.fit(X_train_xgb, y_train)

# Предсказание и оценка
xgb_model.fit(X_train_xgb, y_train_log)
xgb_preds_log = xgb_model.predict(X_test_xgb)
print(f"XGBoost RMSLE: {root_mean_squared_error(y_test_log, xgb_preds_log):.4f}")

In [ ]:
# СОБИРАЕМ PIPELINE

In [ ]:
# 1. Трансформер для расчета географических признаков
class GeoDistanceTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy() # Защита: гарантируем DataFrame

        lat1, lon1 = np.radians(X['pickup_latitude']), np.radians(X['pickup_longitude'])
        lat2, lon2 = np.radians(X['dropoff_latitude']), np.radians(X['dropoff_longitude'])

        # Гаверсинус
        dlat, dlon = lat2 - lat1, lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        X['distance_haversine_km'] = 6371 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

        # Манхэттен и Евклид
        lat_dist = np.abs(X['pickup_latitude'] - X['dropoff_latitude']) * 111.0
        lon_dist = np.abs(X['pickup_longitude'] - X['dropoff_longitude']) * 111.0 * np.cos(np.radians(X['pickup_latitude']))
        X['distance_manhattan_km'] = lat_dist + lon_dist
        X['distance_euclidean_deg'] = np.sqrt((X['pickup_latitude'] - X['dropoff_latitude'])**2 +
                                               (X['pickup_longitude'] - X['dropoff_longitude'])**2)

        # Геометрия
        X['centroid_latitude'] = (X['pickup_latitude'] + X['dropoff_latitude']) / 2
        X['centroid_longitude'] = (X['pickup_longitude'] + X['dropoff_longitude']) / 2
        X['delta_latitude'] = X['dropoff_latitude'] - X['pickup_latitude']
        X['delta_longitude'] = X['dropoff_longitude'] - X['pickup_longitude']

        # Удаляем исходные координаты
        coord_cols = ['pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude']
        X = X.drop(columns=coord_cols, errors='ignore')
        return X

# 2. Трансформер для кластеризации
class GeoClusteringTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=20):
        self.n_clusters = n_clusters
        self.kmeans = MiniBatchKMeans(n_clusters=self.n_clusters, random_state=42, batch_size=100)

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        # Извлекаем значения через numpy-конвертацию во избежание проблем с индексами
        centroids = X_df[['centroid_latitude', 'centroid_longitude']].to_numpy()
        self.kmeans.fit(centroids)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        centroids = X[['centroid_latitude', 'centroid_longitude']].to_numpy()

        # Предсказываем кластеры
        X['pickup_cluster'] = self.kmeans.predict(centroids)
        X['dropoff_cluster'] = self.kmeans.predict(centroids)
        X['route_cluster_combo'] = X['pickup_cluster'].astype(str) + '_' + X['dropoff_cluster'].astype(str)
        return X

# 3. Трансформер для дат, времени и синергий
class DateTimeFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X['pickup_datetime'] = pd.to_datetime(X['pickup_datetime'])

        X['hour'] = X['pickup_datetime'].dt.hour
        X['day_of_week'] = X['pickup_datetime'].dt.dayofweek
        X['is_weekend'] = X['day_of_week'].isin([5, 6]).astype(int)
        X['minute_of_day'] = X['hour'] * 60 + X['pickup_datetime'].dt.minute
        X['time_of_day'] = pd.cut(X['hour'], bins=[-1, 5, 11, 16, 23], labels=[0, 1, 2, 3]).astype(int)

        # Синергии
        X['route_hour_combo'] = X['route_cluster_combo'] + '_h' + X['hour'].astype(str)

        rush_hour_map = {8: 15, 9: 15, 17: 12, 18: 12, 23: 40, 12: 25}
        X['expected_hourly_speed'] = X['hour'].map(rush_hour_map).fillna(25)
        X['estimated_duration_by_speed'] = (X['distance_haversine_km'] / X['expected_hourly_speed']) * 60

        # Удаляем ненужные строковые хвосты перед подачей в модель
        cols_to_drop = ['pickup_datetime', 'route_cluster_combo', 'store_and_fwd_flag']
        X = X.drop(columns=cols_to_drop, errors='ignore')

        # Приведение типов для XGBoost категорий (только для тех колонок, которые остались)
        cat_cols = ['vendor_id', 'hour', 'day_of_week', 'time_of_day']
        for col in cat_cols:
            if col in X.columns:
                X[col] = X[col].astype('category')

        return X

# 4. Исправленный и защищенный FastTargetEncoder
class FastTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, n_splits=5, smoothing=10):
        self.n_splits = n_splits
        self.smoothing = smoothing
        self.global_mean = None
        self.mapping = None

    def fit(self, X, y):
        # 1. Сбрасываем индексы и превращаем в массивы, полностью игнорируя старую индексацию pandas
        X_df = pd.DataFrame(X).reset_index(drop=True)
        y_arr = np.asarray(y).ravel() # Гарантируем плоский одномерный массив numpy

        self.global_mean = float(y_arr.mean())

        # Определяем имя колонки (на случай, если оно изменилось на числовой индекс)
        col_name = 'route_hour_combo' if 'route_hour_combo' in X_df.columns else X_df.columns[-1]
        feat_vals = X_df[col_name].to_numpy()

        # Теперь длины гарантированно совпадут, так как мы стерли индексы
        df_tmp = pd.DataFrame({'feat': feat_vals, 'target': y_arr})

        # Глобальный маппинг для теста
        stats = df_tmp.groupby('feat')['target'].agg(['count', 'mean'])
        smoothed_vals = (stats['count'] * stats['mean'] + self.smoothing * self.global_mean) / (stats['count'] + self.smoothing)
        self.mapping = smoothed_vals.to_dict()
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        col_name = 'route_hour_combo' if 'route_hour_combo' in X_df.columns else X_df.columns[-1]

        # Записываем результат, сохраняя исходную структуру X
        X_df['TE_route_hour_duration'] = X_df[col_name].map(self.mapping).fillna(self.global_mean)

        if 'route_hour_combo' in X_df.columns:
            X_df = X_df.drop(columns=['route_hour_combo'])
        return X_df

    def fit_transform(self, X, y):
        # 1. Полный сброс индексов для синхронизации X и y
        X_df = pd.DataFrame(X).reset_index(drop=True)
        y_arr = np.asarray(y).ravel()

        self.global_mean = float(y_arr.mean())

        col_name = 'route_hour_combo' if 'route_hour_combo' in X_df.columns else X_df.columns[-1]
        feat_vals = X_df[col_name].to_numpy()

        # Создаем датафрейм — теперь эта строчка СТОПРОЦЕНТНО не упадет
        df_tmp = pd.DataFrame({'feat': feat_vals, 'target': y_arr})

        # Записываем глобальный маппинг (для будущих вызовов .predict())
        stats_global = df_tmp.groupby('feat')['target'].agg(['count', 'mean'])
        smoothed_global = (stats_global['count'] * stats_global['mean'] + self.smoothing * self.global_mean) / (stats_global['count'] + self.smoothing)
        self.mapping = smoothed_global.to_dict()

        # 2. Расчет Out-of-Fold
        oof_predictions = np.zeros(len(X_df))
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(df_tmp):
            train_fold = df_tmp.iloc[train_idx]

            stats = train_fold.groupby('feat')['target'].agg(['count', 'mean'])
            smoothed = (stats['count'] * stats['mean'] + self.smoothing * self.global_mean) / (stats['count'] + self.smoothing)
            fold_mapping = smoothed.to_dict()

            val_feats = df_tmp.iloc[val_idx]['feat']
            oof_predictions[val_idx] = val_feats.map(fold_mapping).fillna(self.global_mean).to_numpy()

        # Возвращаем копию исходного X (но с восстановленным оригинальным индексом pandas, чтобы пайплайн не ругался дальше)
        X_out = pd.DataFrame(X).copy()
        X_out['TE_route_hour_duration'] = oof_predictions

        if 'route_hour_combo' in X_out.columns:
            X_out = X_out.drop(columns=['route_hour_combo'])
        return X_out

In [ ]:
# 1. Загружаем сырые датасеты (в них есть все исходные колонки координат и строк)
raw_train = pd.read_csv("train.csv") # Содержит trip_duration
raw_test = pd.read_csv("test.csv")   # НЕ содержит trip_duration

# 2. Разделяем признаки и таргет на трейне
X_train_raw = raw_train.drop(columns=['trip_duration'])
y_train_raw = raw_train['trip_duration']

In [ ]:
# 1. Берем наш старый пайплайн, но УБИРАЕМ из его конца саму модель xgboost, оставляя только фичи
feature_processing_pipeline = Pipeline(steps=[
    ('geo_distances', GeoDistanceTransformer()),
    ('geo_clustering', GeoClusteringTransformer(n_clusters=10)),
    ('datetime_features', DateTimeFeatureTransformer()),
    # Обратите внимание: тут TargetEncoder обучится на логах автоматически благодаря обертке ниже
    ('target_encoding', FastTargetEncoder(n_splits=5, smoothing=10))
])

# 2. Создаем финальную модель
xgb_regressor = xgb.XGBRegressor(n_estimators=100, max_depth=6, enable_categorical=True, random_state=42)

# 3. Собираем полный конвейер вместе с обработкой фичей
full_pipeline = Pipeline(steps=[
    ('features', feature_processing_pipeline),
    ('model', xgb_regressor)
])

# 4. МАГИЯ: Оборачиваем весь пайплайн в трансформер целевой переменной
# func=np.log1p автоматически применит логарифм к y_train при вызове .fit()
# inverse_func=np.expm1 автоматически вернет предсказания в секунды при вызове .predict()
production_pipeline = TransformedTargetRegressor(
    regressor=full_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# 5. Обучаем на СЫРЫХ данных и СЫРОМ таргете (в секундах)
production_pipeline.fit(X_train_raw, y_train)

# 6. Предсказания на сыром тесте вернутся сразу в СЕКУНДАХ!
final_predictions_seconds = production_pipeline.predict(raw_test)

# 7. Как посчитать RMSLE для такого продакшн-пайплайна:
# Так как предсказания вернулись в секундах, для метрики мы логарифмируем их вручную
from sklearn.metrics import root_mean_squared_error
test_preds_seconds = production_pipeline.predict(X_test_raw)

rmsle = root_mean_squared_error(np.log1p(y_test), np.log1p(test_preds_seconds))
print(f"Production Pipeline RMSLE: {rmsle:.4f}")

In [ ]:
# 3. ОБУЧАЕМ ВЕСЬ ПАЙПЛАЙН ОДНОЙ КОМАНДОЙ
# Пайплайн сам посчитает расстояния, обучит KMeans, соберет часы и обучит TargetEncoder без утечек!
full_pipeline.fit(X_train_raw, y_train_raw)

# 4. ДЕЛАЕМ ПРЕДСКАЗАНИЯ НА СЫРЫХ ТЕСТОВЫХ ДАННЫХ
# Нам не нужно вызывать никаких промежуточных функций для raw_test!
final_predictions = full_pipeline.predict(raw_test)

# Предсказания готовы к сохранению или отправке
print(final_predictions[:5])